# 📐 CPT Geometry Trainer (v1)
Entrenamiento de Percepción Espacial (Evitar obstáculos).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import io

# 1. Datos Incrustados
csv_data = """dx,dy,vx,vy,success
21.06,-91.78,5.00,-5.00,1
29.28,-69.22,5.00,-5.00,1
-44.34,96.79,-5.00,5.00,1
98.61,62.71,-5.00,0.00,1
73.49,-87.32,0.00,5.00,1
69.24,-5.08,-5.00,5.00,0
41.21,91.92,5.00,5.00,1
66.29,-17.25,5.00,-5.00,1
98.93,-83.05,0.00,5.00,1
18.55,-46.84,-5.00,0.00,1
-36.70,93.94,-5.00,0.00,1
8.61,55.04,5.00,5.00,1
98.01,-42.61,5.00,5.00,1
-66.25,-14.68,0.00,5.00,1
25.81,65.49,-5.00,0.00,1
-91.24,-50.79,-5.00,5.00,1
-94.98,26.84,-5.00,0.00,1
-27.86,52.43,5.00,0.00,1
-96.60,88.31,-5.00,-5.00,1
53.83,94.90,5.00,0.00,1
-44.69,-49.97,-5.00,0.00,1
91.31,-11.98,-5.00,5.00,1
-78.13,43.73,5.00,5.00,1
52.85,-6.37,0.00,-5.00,1
-86.07,-62.23,-5.00,0.00,1
-72.44,-72.59,5.00,-5.00,1
21.66,-50.32,0.00,-5.00,1
93.63,44.72,0.00,5.00,1
-11.88,65.18,-5.00,-5.00,1
-69.85,27.64,0.00,-5.00,1
-44.20,-88.28,5.00,-5.00,1
-96.98,-59.22,5.00,5.00,0
-95.67,-60.60,-5.00,-5.00,1
-96.64,23.41,-5.00,-5.00,1
60.56,34.56,0.00,-5.00,1
74.34,-65.85,0.00,5.00,1
-79.92,10.26,5.00,5.00,1
-14.10,97.15,5.00,5.00,1
-85.26,68.33,0.00,-5.00,1
65.50,53.70,-5.00,-5.00,0
56.94,14.72,-5.00,-5.00,0
52.42,-50.76,5.00,0.00,1
79.77,15.48,0.00,5.00,1
-56.67,31.56,5.00,5.00,1
29.31,87.50,-5.00,-5.00,0
74.58,-97.87,-5.00,5.00,1
-59.20,46.79,5.00,5.00,1
66.99,-87.03,5.00,0.00,1
44.92,77.85,0.00,-5.00,1
-37.48,42.50,5.00,5.00,1
-44.96,-33.10,-5.00,-5.00,1
-8.05,73.42,0.00,-5.00,0
-73.54,-66.54,0.00,-5.00,1
54.38,-31.00,-5.00,0.00,0
34.32,61.51,5.00,0.00,1
98.19,15.62,0.00,5.00,1
16.01,-99.70,5.00,-5.00,1
-38.84,-57.59,0.00,-5.00,1
-80.49,49.25,0.00,-5.00,1
-12.25,67.94,-5.00,5.00,1
61.21,-44.04,-5.00,0.00,0
-51.30,-71.48,-5.00,0.00,1
-18.16,71.17,0.00,-5.00,0
-80.28,-87.67,-5.00,0.00,1
29.44,-63.30,-5.00,0.00,1
-60.71,4.34,-5.00,5.00,1
-32.23,60.37,5.00,0.00,1
81.52,99.49,0.00,-5.00,1
11.89,-59.12,-5.00,0.00,1
60.82,96.36,-5.00,-5.00,0
-92.69,-26.12,5.00,0.00,1
-53.57,72.68,-5.00,-5.00,1
-65.92,56.87,0.00,5.00,1
-74.91,-16.46,-5.00,0.00,1
95.18,-2.21,5.00,-5.00,1
-91.56,-3.12,5.00,0.00,0
65.18,-88.01,-5.00,-5.00,1
-37.95,91.62,0.00,5.00,1
-68.74,-40.30,0.00,5.00,1
-59.32,-29.64,0.00,5.00,1
71.71,88.91,5.00,5.00,1
62.50,7.72,0.00,-5.00,1
-10.04,89.83,0.00,5.00,1
70.01,39.99,0.00,5.00,1
-72.12,96.83,5.00,5.00,1
-90.22,95.89,5.00,5.00,1
-61.05,52.95,-5.00,5.00,1
52.80,-55.57,0.00,5.00,1
45.93,47.01,5.00,-5.00,1
-4.20,-70.00,-5.00,-5.00,1
98.39,59.15,-5.00,-5.00,0
34.67,-53.08,-5.00,5.00,0
-58.55,-23.26,0.00,-5.00,1
93.09,-7.93,0.00,-5.00,1
-63.74,-0.21,0.00,-5.00,1
48.94,-68.54,0.00,5.00,1
-62.64,0.20,-5.00,0.00,1
84.99,25.42,-5.00,0.00,0
-74.87,-47.37,5.00,-5.00,1
75.39,25.99,0.00,-5.00,1
-28.87,-63.25,-5.00,-5.00,1
-90.72,33.73,-5.00,0.00,1
11.00,-67.48,-5.00,5.00,0
-52.94,-97.23,-5.00,5.00,1
-86.69,5.32,0.00,-5.00,1
89.54,42.72,5.00,0.00,1
-92.26,37.69,0.00,5.00,1
93.67,-17.49,0.00,-5.00,1
31.73,-90.47,5.00,0.00,1
-19.72,85.56,-5.00,0.00,1
-85.83,-83.19,-5.00,0.00,1
-40.07,-87.59,0.00,5.00,1
97.68,28.35,-5.00,0.00,1
99.87,-31.50,5.00,0.00,1
92.35,-26.46,5.00,0.00,1
-75.55,88.39,5.00,5.00,1
-11.58,80.39,-5.00,-5.00,1
-94.10,-50.36,-5.00,5.00,1
-42.70,29.65,0.00,5.00,1
-73.91,95.29,5.00,-5.00,1
-43.51,-45.68,-5.00,0.00,1
54.27,53.09,-5.00,5.00,1
-36.00,69.93,0.00,-5.00,0
80.08,44.81,0.00,5.00,1
20.08,50.49,0.00,-5.00,1
57.35,15.96,5.00,0.00,1
-49.75,22.20,-5.00,5.00,1
35.60,-87.38,-5.00,5.00,0
-16.20,90.24,-5.00,5.00,1
-26.71,-85.91,-5.00,5.00,1
-78.35,-12.65,-5.00,0.00,1
-69.67,-96.57,-5.00,5.00,1
34.57,-80.87,0.00,5.00,0
93.45,52.90,-5.00,5.00,1
-75.46,75.80,0.00,-5.00,1
82.93,-85.74,5.00,-5.00,1
-82.84,8.88,-5.00,0.00,1
89.82,13.59,5.00,-5.00,1
59.38,59.58,-5.00,5.00,1
-53.74,-51.39,-5.00,0.00,1
98.77,-31.96,5.00,5.00,1
0.45,50.62,-5.00,0.00,1
93.33,-32.89,0.00,-5.00,1
93.88,81.13,5.00,0.00,1
46.67,54.01,0.00,5.00,1
-59.09,-35.97,5.00,-5.00,1
41.47,68.74,0.00,5.00,1
-60.26,73.05,5.00,5.00,1
44.73,64.32,0.00,-5.00,0
90.39,-32.19,0.00,-5.00,1
19.09,88.43,5.00,5.00,1
-75.05,27.19,-5.00,5.00,1
80.59,35.58,5.00,5.00,1
88.65,-58.69,0.00,-5.00,1
-13.60,-61.31,5.00,5.00,0
99.35,-49.99,0.00,-5.00,1
-11.08,-53.28,-5.00,0.00,1
-7.72,70.75,5.00,-5.00,0
26.88,72.01,5.00,-5.00,1
-45.38,-93.50,5.00,-5.00,1
30.76,55.08,-5.00,0.00,1
-63.09,16.17,-5.00,5.00,1
-95.29,-44.41,0.00,-5.00,1
41.40,85.45,5.00,0.00,1
-57.88,5.61,-5.00,-5.00,1
-98.36,73.52,0.00,5.00,1
49.26,27.11,0.00,5.00,1
34.49,-65.41,-5.00,-5.00,1
-9.10,94.86,-5.00,5.00,1
-55.99,83.49,-5.00,5.00,1
-79.34,-37.67,0.00,5.00,1
-84.02,-27.46,0.00,-5.00,1
64.29,-57.50,5.00,-5.00,1
-17.78,99.71,0.00,5.00,1
-27.03,-50.64,5.00,5.00,0
-76.49,-14.93,0.00,5.00,1
69.25,-80.66,5.00,0.00,1
53.98,-21.26,0.00,-5.00,1
-34.86,-37.78,0.00,5.00,1
83.92,12.95,-5.00,0.00,0
-28.87,-84.46,0.00,5.00,0
-97.82,-70.70,0.00,5.00,1
-80.50,-89.44,5.00,-5.00,1
-99.21,-40.88,5.00,5.00,1
57.70,-17.88,-5.00,-5.00,1
-50.06,93.05,0.00,5.00,1
-97.44,-58.43,-5.00,5.00,1
17.46,68.24,5.00,5.00,1
71.95,44.06,0.00,5.00,1
-23.41,74.78,5.00,5.00,1
-40.80,50.67,0.00,-5.00,0
22.43,86.10,-5.00,0.00,1
77.06,75.83,5.00,5.00,1
-97.59,69.91,0.00,-5.00,1
13.11,-87.48,-5.00,-5.00,1
23.00,-96.49,0.00,-5.00,1
71.07,45.76,5.00,-5.00,1
87.53,59.10,-5.00,-5.00,0
-55.44,32.43,0.00,-5.00,1
-23.33,71.86,5.00,0.00,1
8.20,97.09,5.00,0.00,1
0.90,72.71,-5.00,-5.00,1
90.25,-44.84,0.00,-5.00,1
56.03,62.97,0.00,-5.00,1
-85.10,56.24,-5.00,-5.00,1
-56.96,25.03,5.00,5.00,1
82.53,2.58,5.00,-5.00,1
69.96,-83.30,0.00,5.00,1
-43.72,99.01,5.00,0.00,1
78.78,-43.28,0.00,5.00,1
-74.17,-77.59,5.00,0.00,1
48.10,-44.32,0.00,5.00,0
35.82,-81.63,0.00,5.00,0
-6.19,-61.29,-5.00,0.00,1
76.53,-43.69,-5.00,5.00,0
-72.81,-40.40,-5.00,0.00,1
10.29,-97.26,-5.00,0.00,1
82.49,-34.66,-5.00,5.00,0
39.90,-49.93,-5.00,5.00,0
94.17,30.60,5.00,0.00,1
90.09,-29.09,5.00,5.00,1
37.27,95.03,-5.00,5.00,1
1.71,54.98,5.00,5.00,1
25.45,-96.95,-5.00,5.00,1
-7.16,65.84,0.00,5.00,1
80.58,77.66,0.00,5.00,1
-24.95,-47.47,-5.00,-5.00,1
76.31,-74.07,0.00,-5.00,1
-49.66,83.30,5.00,0.00,1
80.55,92.95,5.00,5.00,1
-62.41,-33.66,0.00,-5.00,1
90.08,90.37,5.00,5.00,1
-72.68,-18.00,-5.00,5.00,1
40.93,86.37,0.00,5.00,1
89.88,-75.92,0.00,5.00,1
-84.23,99.76,5.00,-5.00,1
-99.26,-63.89,-5.00,5.00,1
-59.88,13.24,0.00,-5.00,1
88.31,-83.48,5.00,0.00,1
-23.58,95.27,-5.00,0.00,1
98.08,-5.44,-5.00,0.00,0
-85.72,-98.64,-5.00,-5.00,1
-54.54,-9.23,5.00,-5.00,1
-65.45,-31.01,5.00,5.00,0
-76.50,3.72,-5.00,-5.00,1
-6.48,-68.50,0.00,5.00,0
-86.19,-59.07,5.00,0.00,1
-82.83,-4.87,5.00,5.00,1
6.05,70.38,5.00,5.00,1
-82.85,-77.36,0.00,5.00,1
-97.82,66.39,-5.00,0.00,1
48.39,53.03,-5.00,0.00,1
92.77,-46.10,-5.00,0.00,1
-14.63,67.94,5.00,-5.00,0
35.68,62.76,5.00,0.00,1
-43.74,-25.47,0.00,-5.00,1
33.89,-98.61,-5.00,5.00,1
-41.87,55.54,-5.00,-5.00,1
53.55,77.26,5.00,0.00,1
98.48,65.36,5.00,-5.00,1
-77.64,7.94,-5.00,5.00,1
-64.78,43.69,0.00,-5.00,1
11.41,58.34,0.00,5.00,1
87.06,9.22,5.00,0.00,1
47.00,-53.28,0.00,-5.00,1
-65.35,16.97,-5.00,5.00,1
-93.85,-60.48,-5.00,5.00,1
91.00,74.64,5.00,-5.00,1
86.90,-1.40,0.00,5.00,1
-48.45,-97.82,-5.00,-5.00,1
-45.41,77.57,-5.00,0.00,1
-29.87,-41.65,0.00,-5.00,1
-2.79,61.63,5.00,5.00,1
53.57,-36.84,-5.00,-5.00,1
-92.52,-98.92,-5.00,-5.00,1
-57.85,87.79,0.00,5.00,1
84.16,-32.52,0.00,5.00,1
63.19,76.29,-5.00,-5.00,0
43.71,-87.30,0.00,-5.00,1
8.44,50.35,5.00,5.00,1
-18.97,62.91,-5.00,0.00,1
86.91,-24.22,0.00,5.00,1
97.22,-33.25,5.00,5.00,1
37.99,-75.87,5.00,0.00,1
-59.77,-74.41,-5.00,-5.00,1
44.09,-73.67,-5.00,5.00,0
-68.47,-68.95,5.00,0.00,1
88.25,-72.32,0.00,-5.00,1
-60.52,-6.41,5.00,5.00,0
-51.70,-7.70,0.00,5.00,1
-28.72,56.38,5.00,-5.00,0
-13.72,60.98,-5.00,-5.00,1
17.22,-82.64,0.00,5.00,0
-83.59,-52.69,5.00,0.00,1
-74.56,12.66,5.00,0.00,0
69.34,78.51,5.00,5.00,1
7.24,-51.46,-5.00,0.00,1
-56.10,-81.03,5.00,0.00,1
91.68,60.99,5.00,-5.00,1
42.64,59.58,5.00,5.00,1
62.29,31.01,-5.00,-5.00,0
95.55,-4.01,-5.00,-5.00,1
44.17,-58.42,5.00,-5.00,1
58.63,-41.22,-5.00,-5.00,1
-55.58,68.18,0.00,-5.00,1
-78.01,62.38,-5.00,0.00,1
-96.34,27.12,0.00,5.00,1
64.35,82.36,0.00,5.00,1
93.05,54.49,5.00,5.00,1
-39.55,-80.77,5.00,0.00,1
-85.49,-36.38,-5.00,0.00,1
78.66,-8.31,0.00,5.00,1
37.77,-33.26,5.00,-5.00,1
87.29,-87.36,-5.00,5.00,1
61.76,93.89,5.00,-5.00,1
52.10,-85.48,0.00,5.00,1
30.25,96.14,0.00,-5.00,1
99.27,-95.82,-5.00,-5.00,1
30.93,-56.88,-5.00,-5.00,1
-35.88,-87.55,5.00,0.00,1
-28.94,51.03,0.00,-5.00,0
-79.77,56.21,0.00,5.00,1
40.66,-40.76,-5.00,5.00,0
-98.11,-1.26,5.00,0.00,0
-75.51,-47.96,-5.00,5.00,1
59.70,43.32,5.00,5.00,1
76.85,4.85,5.00,-5.00,1
26.87,60.16,-5.00,-5.00,0
79.10,65.66,-5.00,5.00,1
3.58,-85.28,5.00,0.00,1
-29.04,-99.47,5.00,5.00,1
6.31,64.88,-5.00,0.00,1
-64.53,-15.02,-5.00,5.00,1
96.03,-17.84,5.00,0.00,1
77.68,99.52,-5.00,5.00,1
-50.48,-42.13,5.00,-5.00,1
-47.56,-39.10,5.00,0.00,0
77.77,61.00,-5.00,5.00,1
-48.84,-96.19,5.00,5.00,0
-39.47,-68.52,5.00,-5.00,1
-28.69,-53.36,0.00,5.00,0
-78.21,-25.25,5.00,5.00,0
31.58,95.89,5.00,-5.00,1
-81.59,-14.67,5.00,0.00,0
-54.95,0.45,5.00,5.00,1
25.94,49.91,5.00,5.00,1
-36.58,63.81,-5.00,-5.00,1
-92.91,27.71,0.00,-5.00,1
57.18,39.02,5.00,-5.00,1
11.50,-82.02,0.00,-5.00,1
-56.89,9.40,5.00,5.00,1
40.46,-80.86,0.00,-5.00,1
-32.69,42.33,-5.00,5.00,1
-93.74,-32.63,5.00,-5.00,1
32.37,74.54,-5.00,-5.00,0
52.02,-94.77,-5.00,5.00,0
96.36,-18.89,5.00,0.00,1
99.61,-70.72,-5.00,0.00,1
44.44,-24.93,-5.00,0.00,1
-63.22,66.72,-5.00,5.00,1
-79.61,3.83,5.00,5.00,1
-81.51,91.83,5.00,0.00,1
-77.36,-48.04,5.00,0.00,1
37.70,62.73,-5.00,0.00,1
-16.22,-69.62,5.00,-5.00,1
-23.43,-80.82,5.00,5.00,0
89.22,83.71,5.00,5.00,1
5.08,-95.98,5.00,0.00,1
-92.20,-99.39,0.00,-5.00,1
77.64,43.76,-5.00,5.00,1
-59.19,-63.75,0.00,5.00,1
-55.01,-0.03,-5.00,-5.00,1
-99.59,84.51,5.00,0.00,1
84.74,89.33,5.00,-5.00,1
-87.72,-43.44,-5.00,0.00,1
86.17,90.90,5.00,0.00,1
32.13,-46.55,-5.00,5.00,1
71.21,-97.05,-5.00,-5.00,1
-66.99,-70.81,0.00,5.00,1
51.70,-98.98,0.00,5.00,1
42.56,-98.16,5.00,0.00,1
6.85,71.85,-5.00,5.00,1
-5.00,94.38,-5.00,5.00,1
9.21,-81.70,5.00,5.00,1
-74.77,-67.40,5.00,5.00,0
74.26,-21.85,-5.00,-5.00,1
72.78,-54.98,5.00,5.00,1
-84.51,-5.59,0.00,-5.00,1
-93.70,-94.51,5.00,-5.00,1
74.56,17.89,5.00,5.00,1
72.58,48.50,-5.00,5.00,1
-74.18,-8.84,-5.00,-5.00,1
83.68,-58.39,5.00,5.00,1
32.13,-90.19,0.00,-5.00,1
-57.90,-56.87,5.00,-5.00,1
-26.17,-65.43,0.00,-5.00,1
-12.61,67.59,5.00,5.00,1
-77.54,-3.84,-5.00,5.00,1
73.51,-52.43,-5.00,-5.00,1
32.14,-65.57,-5.00,5.00,0
29.06,45.62,0.00,-5.00,1
50.07,-80.05,-5.00,-5.00,1
-15.48,-49.05,0.00,-5.00,1
-91.03,-78.36,5.00,0.00,1
-66.45,-26.92,-5.00,5.00,1
-66.23,68.80,0.00,5.00,1
-32.87,-56.84,-5.00,5.00,1
72.39,-34.71,-5.00,-5.00,1
-97.72,-43.56,-5.00,5.00,1
-51.72,-5.68,-5.00,5.00,1
36.23,-73.86,0.00,5.00,0
66.96,-28.31,-5.00,5.00,0
-96.73,-52.49,-5.00,-5.00,1
1.03,74.61,-5.00,5.00,1
31.45,90.95,-5.00,5.00,1
64.20,-71.00,-5.00,0.00,1
-7.88,79.77,-5.00,-5.00,1
71.91,21.63,5.00,-5.00,1
59.25,65.36,0.00,-5.00,1
-90.48,-29.45,5.00,0.00,1
20.07,-87.77,5.00,5.00,1
50.43,-28.20,-5.00,5.00,0
3.13,-87.90,-5.00,-5.00,1
99.14,-78.21,5.00,5.00,1
-28.54,-91.17,5.00,5.00,0
-50.59,5.14,-5.00,5.00,1
-80.90,-58.38,5.00,5.00,0
0.26,53.65,5.00,0.00,1
-96.20,22.42,0.00,-5.00,1
-6.88,-92.99,-5.00,5.00,1
64.63,-27.51,5.00,5.00,1
-28.29,71.16,5.00,0.00,1
88.12,-68.41,0.00,5.00,1
81.46,93.11,-5.00,0.00,1
-97.74,-26.27,5.00,-5.00,1
-95.06,-6.16,5.00,0.00,0
-72.61,33.84,-5.00,5.00,1
54.10,91.76,0.00,5.00,1
12.73,77.30,-5.00,-5.00,0
-38.48,40.86,0.00,-5.00,0
33.98,88.14,5.00,0.00,1
81.22,63.91,-5.00,0.00,1
-36.11,82.75,0.00,-5.00,0
-47.06,-69.20,5.00,5.00,0
95.33,91.82,-5.00,-5.00,1
82.17,-30.83,-5.00,-5.00,1
62.48,-93.13,-5.00,5.00,0
17.24,95.23,-5.00,-5.00,1
-30.75,-43.27,-5.00,-5.00,1
-53.93,-78.68,0.00,-5.00,1
86.13,84.30,-5.00,5.00,1
78.35,-64.79,5.00,0.00,1
-90.31,-51.36,5.00,5.00,0
54.43,32.62,5.00,5.00,1
-14.73,-78.32,5.00,0.00,1
-97.56,-88.03,-5.00,0.00,1
52.56,57.37,0.00,-5.00,1
50.56,45.42,-5.00,0.00,0
60.67,-68.97,5.00,0.00,1
-73.55,-32.21,-5.00,-5.00,1
46.08,-97.74,-5.00,-5.00,1
-15.13,-75.95,5.00,0.00,1
-68.81,-74.11,5.00,-5.00,1
-71.49,45.77,0.00,5.00,1
-60.96,3.90,0.00,5.00,1
30.21,-47.60,0.00,5.00,0
-63.91,92.63,-5.00,5.00,1
-73.85,-53.88,0.00,-5.00,1
-70.29,16.80,-5.00,0.00,1
90.93,96.74,5.00,0.00,1
20.07,83.18,0.00,5.00,1
-25.29,44.22,-5.00,5.00,1
69.95,91.35,0.00,5.00,1
-75.98,-15.18,-5.00,-5.00,1
-13.87,-94.18,-5.00,0.00,1
55.31,-39.53,-5.00,0.00,0
54.08,-79.58,0.00,5.00,1
-88.89,-29.39,0.00,5.00,1
97.64,50.00,0.00,-5.00,1
-95.35,73.35,5.00,-5.00,1
39.55,55.24,5.00,5.00,1
25.33,91.03,0.00,5.00,1
-88.47,-37.51,5.00,5.00,0
45.04,-24.35,0.00,5.00,1
47.67,18.00,-5.00,-5.00,1
61.96,-90.26,-5.00,5.00,0
68.71,-63.56,-5.00,-5.00,1
-3.31,-82.10,-5.00,0.00,1
-72.97,-51.09,-5.00,5.00,1
32.09,-95.89,-5.00,0.00,1
97.58,-92.87,-5.00,5.00,1
66.94,34.39,-5.00,-5.00,0
75.30,-19.23,0.00,-5.00,1
-56.38,-37.27,-5.00,-5.00,1
-80.07,16.57,5.00,5.00,1
14.47,89.73,-5.00,-5.00,1
12.67,96.59,-5.00,0.00,1
-72.39,20.25,-5.00,5.00,1
59.28,87.54,0.00,5.00,1
-75.63,-29.97,5.00,0.00,0
49.39,-87.15,0.00,-5.00,1
63.10,26.46,5.00,-5.00,1
-43.06,-53.47,0.00,-5.00,1
29.30,-57.36,5.00,5.00,1
-61.28,-93.25,5.00,5.00,0
-94.93,-32.00,-5.00,-5.00,1
-24.57,-88.66,-5.00,-5.00,1
-76.01,65.63,5.00,-5.00,0
-31.88,-74.19,0.00,5.00,0
59.85,0.98,0.00,5.00,1
69.28,-74.71,5.00,-5.00,1
-84.47,-82.52,0.00,-5.00,1
66.41,-66.61,-5.00,-5.00,1
70.54,-24.09,-5.00,5.00,0
-73.34,-88.19,5.00,0.00,1
-68.37,-65.03,-5.00,-5.00,1
-69.18,-84.30,5.00,5.00,0
-2.96,94.12,5.00,5.00,1
22.59,-68.64,5.00,-5.00,1
-53.70,84.95,-5.00,0.00,1
40.89,-97.21,0.00,-5.00,1
-29.19,99.33,5.00,5.00,1
71.56,94.11,-5.00,0.00,1
-76.32,-30.83,-5.00,0.00,1
35.84,79.60,5.00,0.00,1
82.67,-75.15,0.00,5.00,1
-69.43,74.82,0.00,5.00,1
51.98,43.48,0.00,-5.00,1
74.57,-9.43,0.00,-5.00,1
-0.09,95.89,-5.00,-5.00,1
42.73,92.70,-5.00,-5.00,0
43.31,-89.21,0.00,-5.00,1
65.22,-50.59,5.00,0.00,1
96.54,48.92,-5.00,0.00,1
-30.26,-59.89,0.00,-5.00,1
17.05,56.02,0.00,-5.00,0
55.10,-36.84,0.00,5.00,1
-42.81,54.80,5.00,5.00,1
-50.39,-45.27,0.00,-5.00,1
62.80,-12.30,5.00,5.00,1
-7.94,-60.53,-5.00,-5.00,1
48.23,28.10,0.00,5.00,1
96.74,-96.77,5.00,-5.00,1
81.06,4.10,-5.00,0.00,0
54.90,-77.49,5.00,5.00,1
-32.42,-68.11,0.00,-5.00,1
-33.83,38.42,5.00,0.00,1
87.74,52.89,0.00,5.00,1
-49.58,60.30,0.00,-5.00,1
-2.17,81.40,5.00,0.00,1
49.43,39.71,5.00,-5.00,1
-54.52,-76.38,0.00,5.00,1
87.60,97.97,5.00,5.00,1
-46.28,-91.81,-5.00,-5.00,1
-64.25,5.71,0.00,5.00,1
-52.22,58.76,0.00,-5.00,1
96.17,-35.36,-5.00,-5.00,1
47.15,62.35,0.00,5.00,1
-59.46,98.58,5.00,0.00,1
41.61,-37.39,5.00,5.00,1
-98.95,80.06,5.00,-5.00,1
83.49,-35.12,5.00,5.00,1
-70.05,47.58,-5.00,-5.00,1
-58.04,-53.23,5.00,0.00,1
88.24,-75.78,0.00,5.00,1
-90.60,47.48,-5.00,-5.00,1
-43.94,-87.19,0.00,5.00,1
94.62,-71.77,5.00,0.00,1
86.76,-94.63,-5.00,5.00,1
-3.57,-79.04,-5.00,5.00,1
50.99,-5.12,-5.00,-5.00,1
94.01,71.00,5.00,5.00,1
67.84,-66.93,-5.00,5.00,0
55.90,98.45,5.00,0.00,1
45.76,-56.14,5.00,5.00,1
36.52,-88.91,0.00,-5.00,1
86.10,-42.31,-5.00,0.00,1
-48.57,-17.31,0.00,-5.00,1
87.33,61.66,5.00,5.00,1
83.44,-39.71,-5.00,5.00,0
-60.02,-22.75,5.00,-5.00,1
89.01,-39.64,-5.00,-5.00,1
32.95,47.88,-5.00,5.00,1
-80.03,-85.27,0.00,5.00,1
-66.24,92.46,-5.00,-5.00,1
48.66,-91.41,5.00,5.00,1
-76.29,-51.00,0.00,5.00,1
47.12,76.39,-5.00,5.00,1
88.24,2.11,5.00,5.00,1
94.76,87.81,-5.00,0.00,1
-76.36,63.80,5.00,-5.00,0
-46.90,-46.49,5.00,5.00,0
29.94,67.42,5.00,5.00,1
80.61,-43.21,5.00,-5.00,1
98.15,2.54,0.00,5.00,1
-29.31,-44.44,5.00,0.00,0
97.80,51.69,-5.00,5.00,1
-84.62,7.58,5.00,-5.00,1
-46.09,59.81,5.00,0.00,1
77.61,87.37,5.00,-5.00,1
24.51,-49.78,-5.00,-5.00,1
84.91,96.68,0.00,5.00,1
-50.94,-74.55,-5.00,0.00,1
-92.06,91.60,-5.00,0.00,1
-91.87,-4.05,0.00,5.00,1
56.69,74.81,-5.00,-5.00,0
-53.71,93.01,-5.00,0.00,1
-16.14,48.39,5.00,0.00,1
37.40,91.31,-5.00,-5.00,0
13.45,-62.18,5.00,5.00,1
18.08,-66.82,-5.00,0.00,1
8.75,99.48,0.00,-5.00,1
-66.12,-51.13,5.00,-5.00,1
-90.92,28.90,-5.00,-5.00,1
84.30,23.98,-5.00,-5.00,0
76.67,57.02,0.00,5.00,1
-57.69,-88.16,-5.00,5.00,1
-90.79,53.64,-5.00,-5.00,1
-82.15,43.48,5.00,-5.00,0
83.90,45.85,0.00,5.00,1
-25.95,43.49,5.00,-5.00,1
53.15,-46.29,-5.00,-5.00,1
-23.69,-61.93,5.00,5.00,0
-65.19,-43.21,0.00,5.00,1
-77.82,-98.81,5.00,5.00,1
-65.45,-28.00,0.00,-5.00,1
-58.91,-46.58,-5.00,0.00,1
48.38,-89.96,5.00,0.00,1
61.25,98.44,5.00,0.00,1
97.77,89.94,-5.00,-5.00,1
45.04,35.03,-5.00,0.00,0
16.11,79.21,-5.00,5.00,1
77.44,-99.46,5.00,0.00,1
70.81,-97.95,-5.00,5.00,1
-94.63,45.54,-5.00,0.00,1
33.96,-79.72,5.00,-5.00,1
73.82,-45.94,0.00,5.00,1
87.33,-32.67,5.00,-5.00,1
81.97,81.29,0.00,5.00,1
-50.29,-39.49,-5.00,0.00,1
61.32,-29.81,0.00,-5.00,1
-98.42,61.90,-5.00,0.00,1
-42.36,-85.72,5.00,-5.00,1
40.21,-88.40,0.00,-5.00,1
-16.65,-92.39,-5.00,-5.00,1
93.69,2.79,5.00,-5.00,1
84.24,56.24,5.00,0.00,1
-3.01,60.77,0.00,5.00,1
31.28,-49.59,5.00,-5.00,1
34.52,95.56,-5.00,5.00,1
-56.43,48.17,5.00,5.00,1
59.90,68.39,-5.00,0.00,1
-48.35,-40.74,5.00,-5.00,1
62.42,-66.43,5.00,0.00,1
-84.07,44.52,0.00,5.00,1
-67.42,-41.16,-5.00,-5.00,1
-47.21,-21.52,0.00,5.00,1
-93.04,-75.73,-5.00,-5.00,1
21.97,-95.35,5.00,0.00,1
63.20,95.39,0.00,5.00,1
2.80,98.55,-5.00,-5.00,1
69.70,-74.22,-5.00,-5.00,1
22.48,-55.60,5.00,-5.00,1
79.37,-73.73,5.00,5.00,1
-98.36,44.68,0.00,5.00,1
-39.48,-56.63,-5.00,5.00,1
-50.74,61.35,0.00,-5.00,1
-22.18,90.78,-5.00,5.00,1
64.46,81.35,0.00,-5.00,1
-5.54,-90.91,5.00,5.00,1
43.24,91.33,5.00,-5.00,1
-70.05,14.38,5.00,-5.00,0
-80.62,-74.72,0.00,-5.00,1
49.92,43.83,5.00,-5.00,1
-85.74,64.79,0.00,5.00,1
50.13,83.09,5.00,0.00,1
62.56,-22.80,-5.00,0.00,0
68.14,50.17,-5.00,-5.00,0
89.99,-56.07,5.00,0.00,1
-21.53,-89.31,5.00,-5.00,1
-88.84,97.55,5.00,5.00,1
95.85,99.37,0.00,-5.00,1
-93.26,-9.34,-5.00,5.00,1
-17.17,-99.15,-5.00,5.00,1
-15.19,55.81,-5.00,5.00,1
-71.11,4.81,5.00,0.00,0
73.51,-14.05,5.00,5.00,1
-42.74,-57.09,0.00,5.00,0
-41.07,32.59,-5.00,0.00,1
24.12,-84.56,5.00,0.00,1
0.66,63.49,-5.00,-5.00,1
-53.64,34.91,-5.00,-5.00,1
-92.57,59.01,0.00,-5.00,1
-62.13,-47.37,-5.00,5.00,1
60.01,-0.24,-5.00,-5.00,1
46.62,76.29,5.00,5.00,1
-76.04,80.56,-5.00,5.00,1
-85.46,-54.75,0.00,-5.00,1
-60.16,-33.35,5.00,-5.00,1
-85.52,-30.67,-5.00,-5.00,1
-60.54,73.53,5.00,-5.00,0
-13.42,-69.47,5.00,0.00,1
68.05,-77.09,5.00,0.00,1
54.05,71.57,0.00,5.00,1
49.14,12.58,-5.00,-5.00,1
-44.30,66.53,5.00,-5.00,0
-19.62,-75.81,-5.00,-5.00,1
-88.68,-83.97,5.00,0.00,1
-59.19,-54.77,5.00,5.00,0
-64.34,-88.70,5.00,0.00,1
52.37,33.58,-5.00,-5.00,0
-25.16,73.93,0.00,-5.00,0
-39.35,47.19,5.00,0.00,0
29.26,-83.46,5.00,0.00,1
19.69,-72.97,5.00,0.00,1
-76.10,-21.59,-5.00,0.00,1
-48.15,67.88,5.00,-5.00,1
16.38,-65.41,-5.00,5.00,1
-72.85,-33.89,0.00,5.00,1
-75.40,37.24,-5.00,5.00,1
57.41,-49.96,-5.00,-5.00,1
-89.91,-13.98,0.00,-5.00,1
-53.44,69.56,5.00,5.00,1
-99.01,55.52,-5.00,0.00,1
24.79,-88.66,-5.00,-5.00,1
63.86,-79.04,0.00,-5.00,1
5.17,53.78,-5.00,-5.00,1
-71.61,-20.16,-5.00,-5.00,1
60.76,-81.68,-5.00,0.00,1
8.07,81.23,-5.00,0.00,1
59.19,-83.78,0.00,5.00,1
74.75,32.39,0.00,5.00,1
83.27,6.98,-5.00,-5.00,1
-84.49,-56.36,-5.00,-5.00,1
-94.51,-0.60,5.00,5.00,1
24.10,85.49,0.00,5.00,1
-89.92,-38.59,5.00,0.00,1
52.47,98.57,-5.00,-5.00,0
-40.19,-69.70,0.00,5.00,0
-88.74,23.94,-5.00,0.00,1
23.28,-85.05,5.00,-5.00,1
72.78,-55.89,-5.00,-5.00,1
-8.06,79.88,5.00,-5.00,1
73.60,-56.84,5.00,0.00,1
-24.92,-88.49,0.00,5.00,0
-55.15,45.48,0.00,5.00,1
83.97,1.60,5.00,5.00,1
-55.96,29.66,0.00,-5.00,1
-36.72,41.83,-5.00,0.00,1
96.64,13.52,-5.00,-5.00,1
-67.51,-97.41,0.00,-5.00,1
63.27,-89.67,0.00,5.00,1
32.22,44.38,5.00,0.00,1
-82.81,45.19,-5.00,-5.00,1
53.89,90.90,0.00,-5.00,1
-45.69,75.58,-5.00,5.00,1
71.77,41.44,5.00,-5.00,1
53.23,-85.77,-5.00,-5.00,1
63.11,70.95,5.00,5.00,1
15.28,-97.20,5.00,0.00,1
84.81,6.78,0.00,-5.00,1
29.66,86.03,5.00,-5.00,1
85.56,-97.88,5.00,-5.00,1
-56.93,-12.23,5.00,5.00,0
-59.14,-44.03,5.00,5.00,0
99.52,40.05,-5.00,-5.00,1
-46.60,-49.39,-5.00,5.00,1
10.16,51.69,0.00,5.00,1
-64.72,25.53,5.00,-5.00,0
39.97,-81.04,5.00,5.00,1
-88.83,-53.77,5.00,-5.00,1
-49.01,87.19,0.00,-5.00,1
7.51,54.54,-5.00,-5.00,1
42.69,-35.21,5.00,5.00,1
60.82,28.36,-5.00,5.00,1
18.65,-48.55,0.00,5.00,1
39.95,-61.88,5.00,0.00,1
3.14,-68.21,5.00,0.00,1
-8.63,-50.58,5.00,0.00,1
15.55,-94.76,0.00,5.00,0
-93.95,66.83,-5.00,5.00,1
-31.04,95.78,-5.00,-5.00,1
-79.13,-8.55,5.00,5.00,1
-58.04,-38.29,-5.00,-5.00,1
-79.87,53.31,5.00,5.00,1
21.23,-54.21,-5.00,5.00,0
-60.48,97.42,5.00,5.00,1
79.69,89.66,-5.00,5.00,1
20.29,61.03,-5.00,-5.00,0
81.13,99.84,5.00,-5.00,1
-38.19,57.23,0.00,5.00,1
3.00,53.57,-5.00,0.00,1
-42.04,-59.33,5.00,0.00,1
-67.93,76.29,-5.00,5.00,1
59.41,-17.82,-5.00,0.00,0
-99.57,-4.30,0.00,5.00,1
-68.75,-53.59,5.00,0.00,1
-79.39,68.50,5.00,-5.00,0
52.65,13.66,5.00,5.00,1
-60.42,92.32,-5.00,5.00,1
1.45,57.77,0.00,-5.00,0
-47.85,-90.04,0.00,5.00,1
-33.42,-50.14,5.00,5.00,0
38.97,-95.82,5.00,5.00,1
29.02,-91.78,-5.00,-5.00,1
-76.53,-97.96,-5.00,-5.00,1
-64.42,-86.95,-5.00,-5.00,1
70.08,74.28,0.00,5.00,1
-31.37,-67.07,0.00,-5.00,1
-61.40,42.14,5.00,-5.00,0
83.52,-49.30,5.00,5.00,1
-85.04,74.20,0.00,5.00,1
62.88,81.19,-5.00,0.00,1
-76.99,79.58,0.00,5.00,1
9.56,-62.47,5.00,5.00,1
-9.99,-53.89,0.00,-5.00,1
52.82,98.78,0.00,5.00,1
-66.30,-17.70,-5.00,-5.00,1
58.01,1.19,-5.00,0.00,0
-30.63,-46.74,5.00,-5.00,1
36.17,98.20,-5.00,0.00,1
-71.91,-79.21,-5.00,5.00,1
-4.23,92.07,5.00,5.00,1
72.53,-88.00,5.00,-5.00,1
62.65,-84.14,5.00,-5.00,1
-55.23,10.45,5.00,-5.00,0
-2.45,79.48,-5.00,-5.00,1
-24.38,-46.11,0.00,-5.00,1
-67.88,4.51,-5.00,5.00,1
91.45,18.47,-5.00,0.00,0
30.88,44.37,0.00,5.00,1
-68.22,-64.53,5.00,-5.00,1
-65.52,49.03,-5.00,5.00,1
-78.92,-20.14,5.00,5.00,0
47.77,-63.10,-5.00,5.00,0
70.98,-23.34,5.00,5.00,1
70.90,12.09,-5.00,-5.00,0
63.43,13.48,5.00,5.00,1
50.13,-92.62,5.00,-5.00,1
42.92,77.16,5.00,-5.00,1
-52.21,52.91,0.00,-5.00,1
-87.66,63.67,5.00,5.00,1
-12.16,69.75,5.00,-5.00,0
-49.72,-46.47,-5.00,0.00,1
92.05,-31.76,0.00,-5.00,1
-22.68,67.12,5.00,-5.00,0
-84.82,-96.32,5.00,0.00,1
-6.47,88.82,0.00,-5.00,0
64.12,40.56,0.00,-5.00,1
-47.43,-25.99,5.00,-5.00,1
-59.20,-29.35,0.00,5.00,1
25.78,87.73,-5.00,-5.00,0
-12.43,87.97,-5.00,0.00,1
92.12,-22.78,5.00,-5.00,1
84.35,-10.49,0.00,-5.00,1
83.44,-74.16,-5.00,-5.00,1
-16.72,52.16,-5.00,0.00,1
96.27,74.60,-5.00,0.00,1
87.69,-56.83,-5.00,0.00,1
48.19,95.38,-5.00,5.00,1
-83.44,-34.40,5.00,5.00,0
-10.36,76.32,5.00,5.00,1
-39.43,43.00,0.00,-5.00,0
-24.27,44.92,0.00,5.00,1
-84.66,-39.91,5.00,5.00,0
24.38,-79.53,5.00,5.00,1
51.84,-94.74,0.00,-5.00,1
97.03,-40.93,0.00,5.00,1
93.88,16.00,5.00,-5.00,1
51.17,-33.51,0.00,-5.00,1
48.57,-63.42,0.00,-5.00,1
-29.79,-62.06,5.00,0.00,1
99.07,-89.42,-5.00,5.00,1
83.08,73.04,5.00,0.00,1
96.63,1.71,0.00,-5.00,1
-18.88,-88.65,5.00,-5.00,1
-99.54,-7.03,5.00,-5.00,1
70.47,-98.44,0.00,5.00,1
62.64,37.31,-5.00,0.00,0
30.74,72.23,-5.00,-5.00,0
-79.39,-18.44,-5.00,0.00,1
62.87,-80.53,-5.00,-5.00,1
75.44,59.77,-5.00,0.00,1
13.81,57.12,-5.00,0.00,1
9.45,55.46,-5.00,5.00,1
-36.32,64.79,0.00,5.00,1
6.26,62.12,5.00,5.00,1
-97.96,-98.66,0.00,-5.00,1
-18.66,76.99,0.00,-5.00,0
78.27,98.07,5.00,0.00,1
-44.78,42.30,5.00,5.00,1
-44.17,-72.90,-5.00,5.00,1
35.56,-88.89,0.00,-5.00,1
-78.06,-9.45,-5.00,0.00,1
-50.70,-73.93,0.00,-5.00,1
11.41,62.95,-5.00,5.00,1
-88.62,-98.43,-5.00,5.00,1
-91.30,33.48,0.00,-5.00,1
76.51,51.19,0.00,-5.00,1
67.89,-72.79,5.00,5.00,1
-52.70,27.16,-5.00,-5.00,1
96.69,96.82,0.00,5.00,1
77.58,-61.05,0.00,-5.00,1
-47.40,-24.06,-5.00,-5.00,1
-7.90,60.73,-5.00,5.00,1
-57.67,-2.39,0.00,5.00,1
-63.96,16.89,5.00,0.00,0
3.28,-71.44,-5.00,0.00,1
-87.39,13.62,-5.00,5.00,1
67.25,-55.56,0.00,-5.00,1
94.05,49.11,0.00,-5.00,1
-38.59,41.23,5.00,0.00,0
-65.53,11.86,5.00,0.00,0
54.07,0.66,5.00,0.00,1
97.35,-71.30,-5.00,-5.00,1
65.50,-10.83,-5.00,-5.00,1
84.01,12.84,0.00,5.00,1
48.35,-41.64,0.00,-5.00,1
-79.63,-90.65,5.00,5.00,1
44.53,-84.16,5.00,5.00,1
65.05,-57.20,5.00,5.00,1
82.90,-78.18,5.00,0.00,1
-79.78,-53.82,0.00,-5.00,1
-56.32,-17.68,5.00,-5.00,1
55.72,-77.51,5.00,5.00,1
89.22,-13.86,5.00,5.00,1
10.06,-82.92,-5.00,0.00,1
-92.21,95.20,5.00,0.00,1
-35.75,69.57,-5.00,-5.00,1
-97.35,1.73,-5.00,-5.00,1
-21.71,75.43,-5.00,5.00,1
54.88,-46.33,0.00,5.00,1
-28.10,74.53,-5.00,-5.00,1
-70.35,-28.62,5.00,5.00,0
85.44,-6.27,5.00,5.00,1
-67.09,75.52,-5.00,-5.00,1
89.85,20.74,-5.00,-5.00,0
-80.17,-67.61,-5.00,0.00,1
36.07,98.57,0.00,-5.00,1
72.85,-90.31,5.00,0.00,1
73.07,52.28,0.00,5.00,1
-76.39,20.77,5.00,-5.00,0
12.51,-78.83,-5.00,5.00,0
73.27,99.33,-5.00,-5.00,1
24.93,74.55,-5.00,5.00,1
92.57,-85.54,5.00,5.00,1
-3.45,-83.96,0.00,-5.00,1
50.39,62.47,5.00,-5.00,1
-95.79,8.97,-5.00,0.00,1
-31.10,-98.52,5.00,0.00,1
99.84,-75.58,0.00,-5.00,1
59.40,-37.00,-5.00,0.00,0
-25.65,-79.47,-5.00,-5.00,1
66.46,-84.41,5.00,5.00,1
-4.57,96.59,-5.00,0.00,1
-10.56,96.44,-5.00,5.00,1
49.45,16.18,5.00,0.00,1
22.96,-46.90,0.00,5.00,1
-62.07,-91.06,0.00,-5.00,1
-62.88,40.86,-5.00,5.00,1
44.71,-29.03,-5.00,0.00,1
-78.17,-58.43,-5.00,0.00,1
90.38,-23.28,0.00,5.00,1
98.93,48.91,-5.00,0.00,1
-94.80,-73.16,0.00,-5.00,1
-93.40,9.10,-5.00,5.00,1
60.15,-81.66,5.00,5.00,1
6.65,-75.62,-5.00,-5.00,1
14.42,97.65,5.00,5.00,1
94.95,2.98,-5.00,5.00,1
-87.31,-48.37,-5.00,0.00,1
72.61,61.50,5.00,-5.00,1
72.03,-58.45,-5.00,-5.00,1
61.44,96.00,-5.00,5.00,1
89.05,14.96,5.00,-5.00,1
89.18,58.07,-5.00,5.00,1
13.28,-61.03,5.00,5.00,1
63.36,-27.88,5.00,0.00,1
-44.41,-84.14,0.00,-5.00,1
-52.13,46.21,-5.00,0.00,1
94.01,-9.00,5.00,0.00,1
75.66,43.00,5.00,0.00,1
-34.08,-45.71,-5.00,5.00,1
-86.08,0.10,5.00,5.00,1
4.23,-89.39,-5.00,0.00,1
73.26,57.91,-5.00,0.00,1
-8.54,-82.42,-5.00,5.00,1
-68.15,93.63,-5.00,-5.00,1
97.06,-35.23,-5.00,-5.00,1
-42.77,63.78,-5.00,-5.00,1
15.69,78.40,0.00,5.00,1
25.55,74.14,5.00,5.00,1
29.07,-85.63,-5.00,-5.00,1
-82.08,59.32,-5.00,0.00,1
18.93,-64.40,-5.00,5.00,0
35.76,87.95,-5.00,0.00,1
-49.27,-63.86,-5.00,0.00,1
-74.38,-21.46,5.00,5.00,0
-68.54,-5.09,0.00,-5.00,1
-69.98,-10.00,5.00,0.00,0
"""
df = pd.read_csv(io.StringIO(csv_data))
print(f'Cargadas {len(df)} muestras')

In [ ]:
# 2. Definir Red Estándar CPT
class TabularNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = TabularNet()
opt = optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCELoss()

In [ ]:
# 3. Entrenamiento
X = torch.tensor(df.drop('success', axis=1).values, dtype=torch.float32)
y = torch.tensor(df['success'].values, dtype=torch.float32).view(-1, 1)

for epoch in range(200):
    opt.zero_grad()
    outputs = model(X)
    loss = loss_fn(outputs, y)
    loss.backward()
    opt.step()
    if epoch % 50 == 0: print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

torch.save(model.state_dict(), 'geometry_tabular_filter.pt')
print('✅ Modelo Geométrico Guardado')

In [ ]:
# 4. Descargar / Guardar
import shutil, os
output_file = 'geometry_tabular_filter.pt'
if os.path.exists('/kaggle/working'):
    print('Model deployed.')
    print('✅ Modelo guardado en Kaggle Output')
else:
    try:
        from google.colab import files
        files.download(output_file)
        print('✅ Modelo descargado desde Colab')
    except ImportError:
        print('✅ Modelo guardado localmente')
